# ACE-Step 1.5 — Music Generation (Colab T4)

Generates the film's background music on the Colab GPU.

Run **top to bottom**. All real work lives in `gpu_music.py` (synced into the
exchange folder by `python -m pipeline.run --stage package`); this notebook
only mounts Drive, installs deps, smoke-tests, and runs the packaged job.
Results land in `heritage-exchange/results/` — sync back locally, then
`python -m pipeline.run --resume`.

In [ ]:
# 1. Verify the GPU (Runtime > Change runtime type > T4 GPU)
import torch
assert torch.cuda.is_available(), 'No GPU — enable T4: Runtime > Change runtime type'
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 2. Mount Google Drive and point at the exchange folder
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
EXCHANGE = Path('/content/drive/MyDrive/heritage-exchange')
assert (EXCHANGE / 'pipeline_gpu').is_dir(), (
    'Exchange folder not found. Run `python -m pipeline.run --stage package` '
    'locally, then sync/upload the exchange dir to My Drive as heritage-exchange.'
)

In [ ]:
# 3. Install ACE-Step (~3-5 min). If the release is not on PyPI yet,
#    use the source install line instead.
!pip install -q acestep
# !pip install -q git+https://github.com/ace-step/ACE-Step.git

In [ ]:
# 4. Import the synced generation module
import sys
sys.path.insert(0, str(EXCHANGE / 'pipeline_gpu'))
import gpu_music

In [ ]:
# 5. Smoke test: generate a ~5s clip to verify the install + GPU path
smoke = gpu_music.run_smoke(EXCHANGE)
print(smoke['status'], smoke.get('files') or smoke.get('error'))

In [ ]:
# 6. Run the packaged music job (writes results/<job_id>/ + results.jsonl)
jobs = sorted((EXCHANGE / 'jobs').glob('music-*/job.json'))
assert jobs, 'No music job found — run `python -m pipeline.run --stage package` locally first.'
entry = gpu_music.run_job(jobs[-1], EXCHANGE)
print(entry['status'], entry.get('files') or entry.get('error'))

## Done

Results are in `heritage-exchange/results/`. Wait for Drive to sync them to
your machine (or download the folder manually), then run locally:

```bash
python -m pipeline.run --resume
```